# 05 - Exportar a TFLite

Convierte ambos modelos a `.tflite` con cuantizacion int8 dinamica. Verifica tamaño y prueba inferencia.


In [ ]:
!pip install -q tensorflow

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

OUT = Path("./outputs")

# Cargar modelos originales (pueden estar en mixed_float16 si se entrenaron así)
tf.keras.mixed_precision.set_global_policy("float32")
m1_orig = tf.keras.models.load_model(OUT / "model1_binary.keras")
m2_orig = tf.keras.models.load_model(OUT / "model2_pathogen.keras")
print("Modelos originales cargados")
print("M1 layers:", len(m1_orig.layers))
print("M2 layers:", len(m2_orig.layers))


In [ ]:
def reconstruir_float32(original, num_clases, model_name):
    '''
    Reconstruye el modelo en float32 puro copiando los pesos del original.
    Necesario cuando el modelo fue entrenado con mixed_float16: los pesos
    se guardan en float32 pero el grafo tiene operaciones float16 que TFLite
    no puede convertir. Reconstruir en float32 elimina esas operaciones.
    '''
    IMG = (224, 224)
    base = tf.keras.applications.EfficientNetB0(
        weights=None, include_top=False, input_shape=(*IMG, 3)
    )
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    if num_clases == 1:
        out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    else:
        out = tf.keras.layers.Dense(num_clases, activation="softmax")(x)
    nuevo = tf.keras.Model(base.input, out, name=model_name)

    # Los pesos del original ya son float32 aunque se computaran en float16
    nuevo.set_weights(original.get_weights())
    print(f"  {model_name}: {len(nuevo.layers)} layers, pesos copiados OK")
    return nuevo


m1 = reconstruir_float32(m1_orig, num_clases=1, model_name="model1_binary")
m2 = reconstruir_float32(m2_orig, num_clases=5, model_name="model2_pathogen")


In [ ]:
def export_tflite(model, path, target_mb=None):
    '''Convierte modelo float32 a TFLite con cuantizacion int8 dinamica.'''
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    print(f"  Convirtiendo {path.name}...")
    tflite_model = converter.convert()

    # Asegurar que el directorio existe
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_bytes(tflite_model)
    size_mb = Path(path).stat().st_size / (1024 * 1024)
    status = ""
    if target_mb:
        ok = size_mb < target_mb
        status = f" [{'OK' if ok else 'GRANDE'}]"
    print(f"    {path.name}: {size_mb:.2f} MB{status}")
    return size_mb


export_tflite(m1, OUT / "model1.tflite", target_mb=5)
export_tflite(m2, OUT / "model2.tflite", target_mb=10)
print("\nExport completado. Copiar a Code/assets/models/")


In [ ]:
def labels_from_indices(path_in, path_out):
    import json
    with open(path_in) as f:
        idx = json.load(f)
    sorted_labels = [k for k, _ in sorted(idx.items(), key=lambda kv: kv[1])]
    Path(path_out).write_text("\n".join(f"{i} {lbl}" for i, lbl in enumerate(sorted_labels)))
    print(f"  {path_out}: {sorted_labels}")


labels_from_indices(OUT / "class_indices_model1_binary.json", OUT / "labels_m1.txt")
labels_from_indices(OUT / "class_indices_model2_pathogen.json", OUT / "labels_m2.txt")


In [ ]:
# Verificar inferencia M1
inter1 = tf.lite.Interpreter(model_path=str(OUT / "model1.tflite"))
inter1.allocate_tensors()
inp1 = inter1.get_input_details()[0]
out1 = inter1.get_output_details()[0]
print("M1 Input:", inp1["shape"], inp1["dtype"])
print("M1 Output:", out1["shape"], out1["dtype"])
dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
inter1.set_tensor(inp1["index"], dummy)
inter1.invoke()
print("M1 salida prueba:", inter1.get_tensor(out1["index"]))

# Verificar inferencia M2
inter2 = tf.lite.Interpreter(model_path=str(OUT / "model2.tflite"))
inter2.allocate_tensors()
inp2 = inter2.get_input_details()[0]
out2 = inter2.get_output_details()[0]
print("M2 Input:", inp2["shape"], inp2["dtype"])
print("M2 Output:", out2["shape"], out2["dtype"])
inter2.set_tensor(inp2["index"], dummy)
inter2.invoke()
print("M2 salida prueba:", inter2.get_tensor(out2["index"]))


## Copiar a Code/

```
Code/assets/models/hs/model.tflite          <- outputs/model1.tflite
Code/assets/models/hs/labels.txt            <- outputs/labels_m1.txt
Code/assets/models/pd/model_unquant.tflite  <- outputs/model2.tflite
Code/assets/models/pd/labels.txt            <- outputs/labels_m2.txt
```
